# 01 — Aprendizaje supervisado con UNSW-NB15
**Equipo 1 — clasificación de cinco ataques y análisis complementario Generic vs resto**

**Integrantes:** Integrante 1 · Integrante 2 · Integrante 3 · Integrante 4  
**Asignatura:** Análisis de Datos · **Periodo:** 2026-2

Notebook independiente y reproducible. Cubre imports, carga y EDA del archivo recibido.

**Alcance honesto:** el CSV no contiene clase `Normal`, ni `proto`, `service` o `state`.
`Generic` es un ataque y nunca se interpreta como benigno.


In [ ]:
from pathlib import Path
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.special import softmax
from IPython.display import display
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, average_precision_score,
    balanced_accuracy_score, calinski_harabasz_score, classification_report,
    completeness_score, confusion_matrix, davies_bouldin_score, f1_score,
    homogeneity_score, normalized_mutual_info_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve, silhouette_score,
    v_measure_score
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, RobustScaler, label_binarize
from sklearn.svm import OneClassSVM, SVC

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 110, "savefig.dpi": 180})

MODE = "supervisado"
ROOT = Path.cwd()
VIS_DIR = ROOT / "visualizaciones" / MODE
RES_DIR = ROOT / "resultados" / MODE
CACHE_DIR = ROOT / ".cache_ml"
for folder in (VIS_DIR, RES_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def locate_dataset(filename="UNSW-NB15_Dataset2.csv"):
    candidates = [ROOT / filename, ROOT / "data" / filename, ROOT.parent / filename]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"No se encontró {filename}. Ubíquelo en la carpeta del notebook, en data/ o en la carpeta padre."
    )


def savefig(name):
    path = VIS_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close()
    return path


def stratified_sample_indices(y, n, random_state=RANDOM_STATE):
    """Devuelve índices posicionales de una muestra estratificada reproducible."""
    y = np.asarray(y)
    if n >= len(y):
        return np.arange(len(y))
    idx, _ = train_test_split(
        np.arange(len(y)), train_size=n, stratify=y, random_state=random_state
    )
    return np.sort(idx)


def tune_f1_threshold(y_true, scores):
    """Ajusta el umbral solo con validación; scores altos significan 'resto de ataques'."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best = int(np.nanargmax(f1))
    return float(thresholds[best]), float(f1[best])


def multiclass_scores(model, X, class_order):
    """Obtiene una matriz comparable de puntuaciones/probabilidades por clase."""
    if hasattr(model, "predict_proba"):
        raw = model.predict_proba(X)
    else:
        raw = softmax(model.decision_function(X), axis=1)
    model_classes = model.classes_
    order = [int(np.where(model_classes == c)[0][0]) for c in class_order]
    return raw[:, order]


def purity_score(y_true, y_cluster):
    tab = pd.crosstab(pd.Series(y_cluster, name="cluster"), pd.Series(y_true, name="clase"))
    return float(tab.max(axis=1).sum() / tab.to_numpy().sum())

print("Entorno preparado.")

## 1. Descripción del problema y preguntas

El objetivo principal es clasificar cada registro en uno de cinco ataques: `Backdoor`, `Probe`,
`Worm`, `Generic` o `DoS`. Como análisis complementario se estudia la separabilidad de `Generic`
frente a los otros cuatro ataques; este experimento permite comparar clasificadores discriminativos
con One-Class SVM, pero **no** mide tráfico normal frente a malicioso.

Preguntas: (1) ¿qué variables de conexión/comportamiento aportan más información? (2) ¿LR o SVM
logra mejor F1 macro sobre la misma prueba? (3) ¿qué clases presentan más errores? (4) ¿qué aporta
One-Class SVM cuando aprende únicamente el patrón de `Generic`?

In [ ]:
t0 = time.time()
DATA_PATH = locate_dataset()
df_raw = pd.read_csv(DATA_PATH)
TARGET = "attack"
FEATURES = [c for c in df_raw.columns if c != TARGET]
EXPECTED_CLASSES = ["Backdoor", "Probe", "Worm", "Generic", "DoS"]

assert TARGET in df_raw.columns
assert len(FEATURES) == 39
assert set(df_raw[TARGET].unique()) == set(EXPECTED_CLASSES)
assert all(pd.api.types.is_numeric_dtype(df_raw[c]) for c in FEATURES)

class_counts_raw = df_raw[TARGET].value_counts()
missing_total = int(df_raw.isna().sum().sum())
inf_total = int(np.isinf(df_raw[FEATURES].to_numpy()).sum())

print(f"Archivo: {DATA_PATH}")
print(f"Dimensión original: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"Variables predictoras numéricas: {len(FEATURES)}")
print(f"Faltantes: {missing_total:,} | Infinitos: {inf_total:,}")
display(pd.DataFrame({"n": class_counts_raw, "%": (100 * class_counts_raw / len(df_raw)).round(2)}))

df = df_raw.copy()

## 2. EDA y análisis de variables disponibles

El archivo recibido **no contiene** `proto`, `service` ni `state`; tampoco contiene clase `Normal`.
No se inventan ni se codifican esas columnas. En su lugar se analizan las variables disponibles:

- **Volumen/tasa:** paquetes, bytes, cargas, pérdidas y medias.
- **Temporización:** duración, intervalos, jitter y tiempos TCP.
- **Estado/conexión:** TTL, ventanas TCP y `ct_state_ttl`.
- **Contexto histórico:** contadores `ct_*` de origen, destino, servicio y puertos.
- **Indicadores de aplicación:** FTP, HTTP, profundidad y longitud de respuesta.

In [ ]:
order = class_counts_raw.index.tolist()
fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette("viridis", len(order))
sns.barplot(x=class_counts_raw.index, y=class_counts_raw.values, hue=class_counts_raw.index, ax=ax, palette=colors, legend=False)
ax.set(title="Distribución de ataques", xlabel="attack", ylabel="Registros")
ax.tick_params(axis="x", rotation=25)
for i, value in enumerate(class_counts_raw.values):
    ax.text(i, value, f"{value:,}\n{value/len(df_raw):.1%}", ha="center", va="bottom", fontsize=9)
savefig("01_eda_distribucion_clases.png")

In [ ]:
quality = pd.DataFrame({
    "indicador": ["Filas", "Faltantes", "Infinitos"],
    "valor": [len(df_raw), missing_total, inf_total]
})
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.barplot(data=quality, x="indicador", y="valor", ax=axes[0], color="#0F766E")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set(title="Calidad del archivo recibido", xlabel="", ylabel="Conteo")
axes[1].axis("off")
quality_text = (
    f"Filas: {len(df_raw):,}\n"
    f"Faltantes: {missing_total:,}\n"
    f"Infinitos: {inf_total:,}"
)
axes[1].text(0.05, 0.8, quality_text, fontsize=15, va="top", linespacing=1.6,
             bbox=dict(boxstyle="round,pad=0.8", facecolor="#ECFDF5", edgecolor="#0F766E"))
savefig("02_eda_calidad_datos.png")

In [ ]:
feature_groups = {
    "volumen_tasa": ["spkts", "dpkts", "sbytes", "dbytes", "rate", "sload", "dload", "sloss", "dloss", "smean", "dmean"],
    "tiempo": ["dur", "sinpkt", "dinpkt", "sjit", "djit", "tcprtt", "synack", "ackdat"],
    "estado_conexion": ["sttl", "dttl", "swin", "dwin", "ct_state_ttl"],
    "contexto": [c for c in FEATURES if c.startswith("ct_") and c != "ct_state_ttl"],
    "aplicacion": ["trans_depth", "response_body_len", "is_ftp_login", "ct_ftp_cmd", "ct_flw_http_mthd", "is_sm_ips_ports"]
}
# Perfil robusto: medianas por clase, estandarizadas por mediana/IQR global.
medians = df.groupby(TARGET)[FEATURES].median()
global_med = df[FEATURES].median()
iqr = (df[FEATURES].quantile(.75) - df[FEATURES].quantile(.25)).replace(0, 1)
robust_profile = ((medians - global_med) / iqr).clip(-5, 5)
profile_features = []
for group, cols in feature_groups.items():
    # dos variables con mayor variación entre clases por grupo
    profile_features += robust_profile[cols].var().nlargest(min(2, len(cols))).index.tolist()
profile_features = list(dict.fromkeys(profile_features))

plt.figure(figsize=(13, 5.5))
sns.heatmap(robust_profile[profile_features], cmap="vlag", center=0, annot=True, fmt=".1f", cbar_kws={"label": "Mediana robusta (IQR)"})
plt.title("Perfiles de conexión y comportamiento por ataque")
plt.xlabel("Variable disponible")
plt.ylabel("Clase")
savefig("03_eda_perfiles_comportamiento.png")
print("Variables destacadas por grupo:", profile_features)

In [ ]:
plot_features = ["dur", "sbytes", "dbytes", "rate", "sttl", "ct_state_ttl"]
sample_idx = stratified_sample_indices(df[TARGET], min(30000, len(df)))
eda_sample = df.iloc[sample_idx].copy()
long = eda_sample.melt(id_vars=TARGET, value_vars=plot_features, var_name="variable", value_name="valor")
long["log1p_valor"] = np.log1p(long["valor"].clip(lower=0))
g = sns.catplot(
    data=long, x=TARGET, y="log1p_valor", col="variable", col_wrap=3,
    kind="box", showfliers=False, sharey=False, height=3.4, aspect=1.2,
    hue=TARGET, palette="viridis", legend=False
)
g.set_axis_labels("Ataque", "log(1 + valor)")
g.set_titles("{col_name}")
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=30)
g.fig.suptitle("Distribuciones robustas de variables clave", y=1.02, fontsize=15)
plt.savefig(VIS_DIR / "04_eda_boxplots_log.png", bbox_inches="tight", dpi=180)
plt.show(); plt.close()

In [ ]:
corr = df[FEATURES].corr(method="spearman")
# Mostrar las 20 variables con mayor correlación media absoluta para conservar legibilidad.
selected_corr = corr.abs().mean().nlargest(20).index
plt.figure(figsize=(12, 10))
sns.heatmap(corr.loc[selected_corr, selected_corr], cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Correlación de Spearman — 20 variables más conectadas")
savefig("05_eda_correlacion_spearman.png")

# Porcentaje de outliers por regla IQR en todos los registros.
q1, q3 = df[FEATURES].quantile(.25), df[FEATURES].quantile(.75)
iqr_raw = q3 - q1
outlier_rate = (((df[FEATURES] < (q1 - 1.5 * iqr_raw)) | (df[FEATURES] > (q3 + 1.5 * iqr_raw))).mean() * 100).sort_values(ascending=False)
display(outlier_rate.head(10).rename("% outliers IQR").to_frame().round(2))

## 3. Partición y preprocesamiento sin fuga

Se reserva 15% para prueba y 15% para validación,
siempre estratificando por `attack`. La prueba no participa en selección de variables, búsqueda de
hiperparámetros ni ajuste de umbral. Mutual Information se calcula exclusivamente con entrenamiento;
después se elige el menor conjunto que acumula 90% de la MI (entre 15 y 30 variables). El escalador
se ajusta con entrenamiento y se reutiliza sin reajuste en validación y prueba.

In [ ]:
X = df[FEATURES].copy()
y_text = df[TARGET].copy()
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = label_encoder.classes_.tolist()

X_train, X_temp, y_train, y_temp, attack_train, attack_temp = train_test_split(
    X, y, y_text, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test, attack_val, attack_test = train_test_split(
    X_temp, y_temp, attack_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

for obj in (X_train, X_val, X_test, attack_train, attack_val, attack_test):
    obj.reset_index(drop=True, inplace=True)
y_train, y_val, y_test = map(np.asarray, (y_train, y_val, y_test))

split_summary = pd.DataFrame({
    "conjunto": ["train", "validación", "prueba"],
    "n": [len(X_train), len(X_val), len(X_test)],
    "porcentaje": [len(X_train)/len(df), len(X_val)/len(df), len(X_test)/len(df)]
})
display(split_summary.style.format({"porcentaje": "{:.1%}"}))

## 4. Selección de variables con Mutual Information

La MI se calcula solo en entrenamiento. El número de variables no depende del número de clases: se
escoge el menor conjunto que concentra 90% de la información mutua observada, acotado entre 15 y 30
para conservar señal suficiente y reducir el costo del SVM.

In [ ]:
mi_sample_idx = stratified_sample_indices(y_train, min(50000, len(y_train)))
mi_values = mutual_info_classif(
    X_train.iloc[mi_sample_idx], y_train[mi_sample_idx], random_state=RANDOM_STATE
)
mi_df = pd.DataFrame({"variable": FEATURES, "MI": mi_values}).sort_values("MI", ascending=False)
mi_positive = mi_df["MI"].clip(lower=0)
mi_cumulative = mi_positive.cumsum() / max(mi_positive.sum(), 1e-12)
k_mi = int(np.searchsorted(mi_cumulative.to_numpy(), 0.90) + 1)
k_mi = int(np.clip(k_mi, 15, 30))
selected_features = mi_df.head(k_mi)["variable"].tolist()
plt.figure(figsize=(10, 7))
sns.barplot(data=mi_df.head(20), x="MI", y="variable", color="#7C3AED")
plt.title("Top 20 variables por información mutua — solo entrenamiento")
plt.xlabel("Mutual Information")
plt.ylabel("")
savefig("06_seleccion_variables_mi.png")
display(mi_df.head(20).reset_index(drop=True))
print(f"Se usarán {k_mi} variables (criterio: 90% de MI, límites 15–30).")
print(selected_features)

## 5. Componente supervisado principal: cinco ataques

**Logistic Regression** es un baseline lineal interpretable, rápido y probabilístico; puede fallar
ante fronteras no lineales. **SVM RBF** modela fronteras complejas y funciona bien con variables
escaladas, pero su costo crece más que linealmente. Por ello, LR usa todo el entrenamiento y SVM
una muestra estratificada de 12.000 registros; ambos se evalúan sobre la misma prueba completa.
`class_weight='balanced'` compensa el desbalance entre ataques. `C` y
`gamma` se eligen por F1 macro con GridSearchCV de 3 folds dentro del entrenamiento; nunca se
optimiza sobre validación ni prueba.

In [ ]:
X_train_sup = X_train[selected_features]
X_val_sup = X_val[selected_features]
X_test_sup = X_test[selected_features]

# GridSearchCV se ajusta únicamente con muestras del entrenamiento; validación y prueba no entran.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
lr_train_idx = stratified_sample_indices(y_train, min(30000, len(y_train)), RANDOM_STATE + 1)
X_train_lr, y_train_lr = X_train_sup.iloc[lr_train_idx], y_train[lr_train_idx]
lr_pipe = Pipeline([
    ("scale", RobustScaler()),
    ("clf", LogisticRegression(max_iter=1400, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE))
])
lr_grid = GridSearchCV(
    lr_pipe, {"clf__C": [0.1, 1.0, 10.0]}, scoring="f1_macro",
    cv=cv, n_jobs=-1, refit=True
)
start = time.time()
lr_grid.fit(X_train_lr, y_train_lr)
best_lr = lr_grid.best_estimator_
lr_val_f1 = f1_score(y_val, best_lr.predict(X_val_sup), average="macro")
print(f"LR GridSearch listo en {(time.time()-start)/60:.1f} min | n_train={len(X_train_lr):,} | {lr_grid.best_params_} | CV F1-macro={lr_grid.best_score_:.4f} | validación={lr_val_f1:.4f}")

# SVM: muestreo estratificado permitido por el tamaño del dataset y el costo O(n²–n³).
svm_train_idx = stratified_sample_indices(y_train, min(12000, len(y_train)), RANDOM_STATE)
X_train_svm, y_train_svm = X_train_sup.iloc[svm_train_idx], y_train[svm_train_idx]
svm_pipe = Pipeline([
    ("scale", RobustScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced", cache_size=1800, random_state=RANDOM_STATE))
])
svm_grid = GridSearchCV(
    svm_pipe,
    {"clf__C": [1.0, 5.0], "clf__gamma": ["scale", 0.05]},
    scoring="f1_macro", cv=cv, n_jobs=-1, refit=True
)
start = time.time()
svm_grid.fit(X_train_svm, y_train_svm)
best_svm = svm_grid.best_estimator_
svm_val_f1 = f1_score(y_val, best_svm.predict(X_val_sup), average="macro")
print(f"SVM GridSearch listo en {(time.time()-start)/60:.1f} min | n_train={len(X_train_svm):,} | {svm_grid.best_params_} | CV F1-macro={svm_grid.best_score_:.4f} | validación={svm_val_f1:.4f}")

In [ ]:
def evaluate_multiclass(name, model, X_eval, y_eval):
    pred = model.predict(X_eval)
    scores = multiclass_scores(model, X_eval, np.arange(len(class_names)))
    y_bin = label_binarize(y_eval, classes=np.arange(len(class_names)))
    return {
        "Modelo": name,
        "Accuracy": accuracy_score(y_eval, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_eval, pred),
        "Precision macro": precision_score(y_eval, pred, average="macro", zero_division=0),
        "Recall macro": recall_score(y_eval, pred, average="macro", zero_division=0),
        "F1 macro": f1_score(y_eval, pred, average="macro", zero_division=0),
        "ROC-AUC OVR macro": roc_auc_score(y_bin, scores, average="macro", multi_class="ovr"),
        "PR-AUC macro": average_precision_score(y_bin, scores, average="macro"),
        "pred": pred,
        "scores": scores,
    }

lr_result = evaluate_multiclass("Logistic Regression", best_lr, X_test_sup, y_test)
svm_result = evaluate_multiclass("SVM RBF", best_svm, X_test_sup, y_test)
model_results = [lr_result, svm_result]
metrics_multiclass = pd.DataFrame([
    {k: v for k, v in r.items() if k not in {"pred", "scores"}} for r in model_results
]).set_index("Modelo")
metrics_multiclass.to_csv(RES_DIR / "metricas_supervisado_multiclase.csv")
display(metrics_multiclass.style.format("{:.4f}"))

for r in model_results:
    print(f"\n{r['Modelo']} — reporte por clase")
    print(classification_report(y_test, r["pred"], target_names=class_names, digits=4, zero_division=0))

In [ ]:
plot_df = metrics_multiclass[["Accuracy", "Balanced Accuracy", "Precision macro", "Recall macro", "F1 macro"]].reset_index().melt(id_vars="Modelo", var_name="Métrica", value_name="Valor")
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_df, x="Métrica", y="Valor", hue="Modelo", palette=["#0F766E", "#7C3AED"])
plt.ylim(0, 1.05)
plt.title("Comparación multiclase en la misma prueba retenida")
plt.xticks(rotation=20)
savefig("07_supervisado_comparacion_metricas.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, r in zip(axes, model_results):
    cm = confusion_matrix(y_test, r["pred"], normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set(title=f"{r['Modelo']} — matriz normalizada", xlabel="Predicción", ylabel="Real")
plt.savefig(VIS_DIR / "08_supervisado_matrices_confusion.png", bbox_inches="tight", dpi=180)
plt.show(); plt.close()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
y_test_bin = label_binarize(y_test, classes=np.arange(len(class_names)))
for ax, r in zip(axes, model_results):
    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], r["scores"][:, i])
        auc = roc_auc_score(y_test_bin[:, i], r["scores"][:, i])
        ax.plot(fpr, tpr, label=f"{cls} ({auc:.3f})")
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set(title=f"ROC one-vs-rest — {r['Modelo']}", xlabel="FPR", ylabel="TPR")
    ax.legend(fontsize=8)
plt.savefig(VIS_DIR / "09_supervisado_curvas_roc_multiclase.png", bbox_inches="tight", dpi=180)
plt.show(); plt.close()

In [ ]:
# Curvas Precision–Recall multiclase one-vs-rest (las áreas exactas también están en la tabla).
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, r in zip(axes, model_results):
    for i, cls in enumerate(class_names):
        p_curve, r_curve, _ = precision_recall_curve(y_test_bin[:, i], r["scores"][:, i])
        ap = average_precision_score(y_test_bin[:, i], r["scores"][:, i])
        ax.plot(r_curve, p_curve, label=f"{cls} ({ap:.3f})")
    ax.set(title=f"Precision–Recall OVR — {r['Modelo']}", xlabel="Recall", ylabel="Precision")
    ax.legend(fontsize=8)
plt.savefig(VIS_DIR / "10_supervisado_curvas_pr_multiclase.png", bbox_inches="tight", dpi=180)
plt.show(); plt.close()